# 07. 고급 기능

## 학습 목표
- Human-in-the-Loop 워크플로를 구현한다
- 다양한 스트리밍 모드와 네임스페이스 시스템을 이해한다
- 샌드박스(Modal, Daytona, Runloop) 연동 개념을 파악한다
- ACP(Agent Client Protocol)로 에디터와 연동하는 방법을 안다
- Deep Agents CLI 사용법을 익힌다

In [ ]:
# 환경 설정
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY가 설정되지 않았습니다!"
print("환경 설정 완료")

In [ ]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault(
        "LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default")
    )
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler

    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1")

print(f"모델 설정 완료: {model.model_name}")

---
## 1. Human-in-the-Loop (HITL)

에이전트가 민감한 도구를 호출할 때 **사람의 승인을 요구**하는 워크플로입니다.

### 작동 방식

![Human-in-the-Loop 워크플로](../assets/images/hitl_flow.png)

### 필수 요구사항
- **Checkpointer**: 중단/재개 사이의 에이전트 상태를 유지하기 위해 반드시 필요

In [ ]:
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import MemorySaver

# interrupt_on으로 승인이 필요한 도구 지정
hitl_agent = create_deep_agent(
    model=model,
    system_prompt="당신은 파일 관리 어시스턴트입니다. 한국어로 응답하세요.",
    checkpointer=MemorySaver(),  # 필수!
    interrupt_on={
        "write_file": True,  # 파일 쓰기 전 승인 필요
        "edit_file": True,  # 파일 편집 전 승인 필요
    },
)

print("Human-in-the-Loop 에이전트 생성 완료")
print("write_file, edit_file 호출 시 승인을 요구합니다.")

In [ ]:
# 에이전트 실행 — write_file 호출 시 중단됨
config = {"configurable": {"thread_id": "hitl-demo"}}

result = hitl_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "config.yaml 파일을 만들어서 'debug: true'라고 적어 주세요.",
            }
        ]
    },
    config={**config, **lf_config},
)

# 인터럽트 확인
if "__interrupt__" in result:
    interrupt_info = result["__interrupt__"]
    print("에이전트가 중단되었습니다!")
    print(f"승인 대기 중인 작업: {len(interrupt_info)}개")
    for item in interrupt_info:
        val = item.value if hasattr(item, "value") else item
        print(f"  - 인터럽트 값: {val}")
else:
    print("중단 없이 실행 완료:")
    print(result["messages"][-1].content)

In [ ]:
from langgraph.types import Command

# 승인하여 재개
if "__interrupt__" in result:
    # approve 결정으로 재개
    resumed = hitl_agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}  # 승인
                    # {"type": "reject"}  # 거부
                    # {"type": "edit", "args": {"content": "debug: false"}}  # 수정
                ]
            }
        ),
        config={**config, **lf_config},  # 같은 thread_id 사용
    )

    print("✅ 승인 후 재개 결과:")
    print(resumed["messages"][-1].content)

---
## 2. 스트리밍 심화

Deep Agents는 LangGraph의 스트리밍 인프라 위에서 작동합니다.

### 스트림 모드

| 모드 | 설명 | 사용 시나리오 |
|------|------|-------------|
| `"updates"` | 각 노드 완료 시 상태 업데이트 | 진행 상황 추적 |
| `"messages"` | 개별 토큰 단위 스트리밍 | 실시간 텍스트 출력 |
| `"custom"` | 도구 내부에서 발행하는 이벤트 | 커스텀 진행률 |

### 네임스페이스 시스템

서브에이전트의 이벤트는 네임스페이스로 구분됩니다:

```python
()                          # 메인 에이전트
("tools:abc123",)           # 서브에이전트 (tool call ID)
("tools:abc123", "model:def456")  # 서브에이전트 내부 노드
```

In [ ]:
from typing import Literal
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY", ""))


def internet_search(
    query: str,
    max_results: int = 3,
    topic: Literal["general", "news"] = "general",
) -> dict:
    """인터넷에서 정보를 검색합니다."""
    return tavily_client.search(query, max_results=max_results, topic=topic)


# 서브에이전트 포함 에이전트
stream_agent = create_deep_agent(
    model=model,
    system_prompt="당신은 리서치 코디네이터입니다. 한국어로 응답하세요.",
    subagents=[
        {
            "name": "researcher",
            "description": "인터넷 검색을 통해 정보를 조사합니다.",
            "system_prompt": "인터넷을 검색하여 요청된 정보를 수집하고 간결하게 요약하세요.",
            "tools": [internet_search],
        }
    ],
)

print("스트리밍 데모 에이전트 생성 완료")

In [ ]:
# 서브그래프 포함 스트리밍 — 네임스페이스로 이벤트 소스 구분
print("=== 서브에이전트 이벤트 스트리밍 ===")
print()

for namespace, chunk in stream_agent.stream(
    {
        "messages": [
            {"role": "user", "content": "LangGraph의 최신 기능을 조사해 주세요."}
        ]
    },
    stream_mode="updates",
    subgraphs=True,
    config=lf_config,
):
    # 네임스페이스로 메인/서브에이전트 구분
    source = "[메인]" if not namespace else f"[서브에이전트: {namespace}]"

    for node_name, node_data in chunk.items():
        if not node_data:
            continue
        msgs = node_data.get("messages", [])
        # LangGraph may wrap state updates in Overwrite objects
        if hasattr(msgs, "value"):
            msgs = msgs.value
        if msgs:
            last_msg = msgs[-1]
            if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
                for tc in last_msg.tool_calls:
                    print(f"{source} 🔧 도구 호출: {tc['name']}")
            elif hasattr(last_msg, "content") and last_msg.content:
                content = (
                    last_msg.content
                    if isinstance(last_msg.content, str)
                    else str(last_msg.content)
                )
                if content.strip() and not hasattr(last_msg, "tool_call_id"):
                    preview = content[:100].replace("\n", " ")
                    print(f"{source} 💬 {preview}...")

In [ ]:
# 복수 스트림 모드 동시 사용
print("=== 복수 스트림 모드 ===")
print()

for event in stream_agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "Python 3.13의 새로운 기능을 한 줄로 설명해 주세요.",
            }
        ]
    },
    stream_mode=["updates", "messages"],
    subgraphs=True,
    config=lf_config,
):
    # subgraphs=True + list stream_mode -> variable tuple length
    *namespace_parts, mode, data = event
    if mode == "updates":
        for node_name in data:
            print(f"  [updates] 노드 '{node_name}' 완료")
    elif mode == "messages":
        msg, metadata = data
        if (
            hasattr(msg, "content")
            and msg.content
            and metadata
            and metadata.get("langgraph_node") == "model"
        ):
            print(msg.content, end="", flush=True)

print()

---
## 3. 샌드박스 (Sandbox)

샌드박스는 에이전트가 **격리된 환경**에서 코드를 실행할 수 있게 합니다.  
호스트 시스템의 파일, 네트워크, 자격 증명에 접근하지 못하므로 안전합니다.

### 지원 프로바이더

| 프로바이더 | 특징 | 적합한 용도 |
|-----------|------|------------|
| **Modal** | GPU 지원, ML 워크로드 | AI/ML 작업 |
| **Daytona** | TypeScript/Python, 빠른 콜드 스타트 | 웹 개발 |
| **Runloop** | 일회용 devbox, 격리 실행 | 코드 테스트 |

### 아키텍처 패턴

**샌드박스를 도구로 사용** (권장)

![샌드박스 아키텍처](../assets/images/sandbox_architecture.png)

### ⚠️ 보안 주의사항
- **절대 샌드박스 안에 시크릿을 넣지 마세요** — 에이전트가 유출할 수 있습니다
- 자격 증명은 외부 도구에서만 관리
- Human-in-the-Loop으로 민감한 작업 승인
- 불필요한 네트워크 접근 차단

In [ ]:
# 샌드박스 연동 코드 예시 (실제 실행하려면 해당 프로바이더 설정 필요)

# Modal 샌드박스 예시
sandbox_example_code = """
# pip install deepagents-modal
from deepagents.backends.sandbox import ModalSandbox

agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    backend=ModalSandbox(
        image="python:3.12-slim",
        gpu="T4",  # GPU 지원
    ),
)
"""

print("샌드박스 연동 코드 예시 (참고용):")
print(sandbox_example_code)

---
## 4. ACP (Agent Client Protocol)

ACP는 **코딩 에이전트와 에디터/IDE 간의 통신을 표준화**하는 프로토콜입니다.

### 지원 에디터
- **Zed** — 네이티브 통합
- **JetBrains IDEs** — 빌트인 지원
- **VS Code** — vscode-acp 플러그인
- **Neovim** — ACP 호환 플러그인

### MCP vs ACP
| 프로토콜 | 용도 |
|---------|------|
| MCP (Model Context Protocol) | 외부 도구 통합 |
| ACP (Agent Client Protocol) | 에디터-에이전트 통합 |

In [ ]:
# ACP 서버 구현 예시 (참고용)
acp_example_code = """
# pip install deepagents-acp
from deepagents import create_deep_agent
from deepagents_acp import AgentServerACP
from langgraph.checkpoint.memory import MemorySaver

# 에이전트 생성
agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    system_prompt="당신은 코딩 어시스턴트입니다.",
    checkpointer=MemorySaver(),
)

# ACP 서버 실행 (stdio 모드)
server = AgentServerACP(agent)
server.run()
"""

print("ACP 서버 구현 예시 (참고용):")
print(acp_example_code)

---
## 5. Deep Agents CLI

SDK 위에 구축된 **터미널 코딩 에이전트**입니다.

### 설치 및 실행
```bash
# 설치
uv tool install deepagents-cli

# 실행
deepagents-cli

# 직접 실행 (설치 없이)
uvx deepagents-cli
```

### 주요 옵션

| 옵션 | 설명 |
|------|------|
| `-a/--agent AGENT` | 에이전트 이름 지정 |
| `-M/--model MODEL` | 모델 선택 |
| `-n/--non-interactive` | 비대화형 모드 (단일 태스크 실행) |
| `--auto-approve` | 인간 확인 스킵 |
| `--sandbox {none,modal,daytona,runloop}` | 샌드박스 백엔드 선택 |

### 인터랙티브 명령어

| 명령 | 설명 |
|------|------|
| `/model` | 모델 변경 |
| `/remember` | 메모리에 정보 저장 |
| `/tokens` | 토큰 사용량 확인 |
| `!command` | 쉘 명령 실행 |

### 메모리 시스템
- **글로벌**: `~/.deepagents/<agent_name>/memories/`
- **프로젝트**: `.deepagents/AGENTS.md` (프로젝트 루트)

In [ ]:
# CLI 비대화형 모드 예시 (셸에서 실행)
cli_examples = """
# 기본 사용
deepagents-cli

# 특정 모델로 비대화형 실행
deepagents-cli -M claude-sonnet-4-6 -n "이 프로젝트의 README.md를 작성해 줘"

# 샌드박스에서 실행
deepagents-cli --sandbox modal "테스트 코드를 실행해 줘"

# 스킬 관리
deepagents-cli skills list
deepagents-cli skills create my-skill
"""

print("CLI 사용 예시 (터미널에서 실행):")
print(cli_examples)

-
-
-

#
#
 
전
체
 
교
육
 
자
료
 
정
리


|
 
노
트
북
 
|
 
주
제
 
|
 
핵
심
 
A
P
I
 
|

|
-
-
-
-
-
-
-
-
|
-
-
-
-
-
-
|
-
-
-
-
-
-
-
-
-
-
|

|
 
*
*
0
1
*
*
 
|
 
소
개
 
|
 
`
d
e
e
p
a
g
e
n
t
s
.
_
_
v
e
r
s
i
o
n
_
_
`
 
|

|
 
*
*
0
2
*
*
 
|
 
퀵
스
타
트
 
|
 
`
c
r
e
a
t
e
_
d
e
e
p
_
a
g
e
n
t
(
)
`
,
 
`
i
n
v
o
k
e
(
)
`
,
 
`
s
t
r
e
a
m
(
)
`
 
|

|
 
*
*
0
3
*
*
 
|
 
커
스
터
마
이
징
 
|
 
`
m
o
d
e
l
`
,
 
`
s
y
s
t
e
m
_
p
r
o
m
p
t
`
,
 
`
t
o
o
l
s
`
,
 
`
r
e
s
p
o
n
s
e
_
f
o
r
m
a
t
`
 
|

|
 
*
*
0
4
*
*
 
|
 
백
엔
드
 
|
 
`
S
t
a
t
e
B
a
c
k
e
n
d
`
,
 
`
F
i
l
e
s
y
s
t
e
m
B
a
c
k
e
n
d
`
,
 
`
S
t
o
r
e
B
a
c
k
e
n
d
`
,
 
`
C
o
m
p
o
s
i
t
e
B
a
c
k
e
n
d
`
 
|

|
 
*
*
0
5
*
*
 
|
 
서
브
에
이
전
트
 
|
 
`
S
u
b
A
g
e
n
t
`
,
 
`
C
o
m
p
i
l
e
d
S
u
b
A
g
e
n
t
`
,
 
`
s
u
b
a
g
e
n
t
s
`
 
|

|
 
*
*
0
6
*
*
 
|
 
메
모
리
 
&
 
스
킬
 
|
 
`
m
e
m
o
r
y
`
,
 
`
s
k
i
l
l
s
`
,
 
`
A
G
E
N
T
S
.
m
d
`
,
 
`
S
K
I
L
L
.
m
d
`
 
|

|
 
*
*
0
7
*
*
 
|
 
고
급
 
기
능
 
|
 
`
i
n
t
e
r
r
u
p
t
_
o
n
`
,
 
`
s
t
r
e
a
m
_
m
o
d
e
`
,
 
S
a
n
d
b
o
x
,
 
A
C
P
,
 
C
L
I
 
|


#
#
#
 
다
음
 
단
계

→
 
*
*
[
0
8
.
 
에
이
전
트
 
하
네
스
]
(
0
8
_
h
a
r
n
e
s
s
.
i
p
y
n
b
)
*
*
으
로
 
진
행
하
세
요
!

→
 
*
*
[
고
급
 
과
정
]
(
.
.
/
0
5
_
a
d
v
a
n
c
e
d
/
0
0
_
m
i
g
r
a
t
i
o
n
.
i
p
y
n
b
)
*
*
으
로
 
건
너
뛰
기